# QLoRA JSON Extraction: SFT + DPO Fine-Tuning

This notebook orchestrates the end-to-end training pipeline. It runs on a free T4 GPU in Google Colab.

In [ ]:
# 1. Install GPU-enabled libraries
!pip install -q "transformers>=4.44" "trl>=0.9" peft bitsandbytes datasets accelerate pyyaml faker pydantic

## Project Setup & Git Sync
Run the cell below. It will automatically clone the repository if it's the first run, or force-sync the latest updates from GitHub if you run it again.

In [ ]:
import os
repo_dir = "/content/json-extract-qlora-dpo"

if not os.path.exists(repo_dir):
    print("Cloning repository fresh...")
    %cd /content
    !git clone https://github.com/Tahleels/json-extract-qlora-dpo.git
    %cd json-extract-qlora-dpo
else:
    print("Repository already exists. Running a clean force-sync...")
    %cd {repo_dir}
    !git fetch origin
    !git reset --hard origin/main
    print("Sync complete! Current folder contains the latest GitHub files.")

## Generate Datasets
Run SFT and DPO synthetic generators locally on the Colab instance.

In [ ]:
!python -m src.data.build_sft
!python -m src.data.build_dpo

In [ ]:
# Confirm datasets generated correctly
assert os.path.exists("data/sft_train.jsonl"), "SFT train set missing!"
assert os.path.exists("data/dpo_train.jsonl"), "DPO train set missing!"

## Stage 2: SFT Training (QLoRA)
We load the base model in 4-bit precision, mount the LoRA adapters, format SFT data with a chat template, and fine-tune.

In [ ]:
import torch
import yaml
from pathlib import Path
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig
from trl import SFTTrainer, SFTConfig
from src.config import load_config

cfg = load_config()
with open("configs/sft.yaml", "r", encoding="utf-8") as f:
    sft_cfg = yaml.safe_load(f)

# 4-bit NF4 Config
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

print(f"Loading base model in 4-bit: {cfg.base_model}")
tokenizer = AutoTokenizer.from_pretrained(cfg.base_model)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

model = AutoModelForCausalLM.from_pretrained(
    cfg.base_model,
    quantization_config=bnb_config,
    device_map="auto",
)

# LoRA Setup
peft_config = LoraConfig(
    r=sft_cfg["lora"]["r"],
    lora_alpha=sft_cfg["lora"]["alpha"],
    lora_dropout=sft_cfg["lora"]["dropout"],
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=sft_cfg["lora"]["target_modules"],
)

# Load datasets
dataset = load_dataset(
    "json",
    data_files={
        "train": str(Path(cfg.data["sft_out_dir"]) / "sft_train.jsonl"),
        "val": str(Path(cfg.data["sft_out_dir"]) / "sft_val.jsonl"),
    }
)

# Apply Chat Template
def format_chat(example):
    messages = [
        {"role": "system", "content": cfg.system_prompt},
        {"role": "user", "content": example["input_text"]},
        {"role": "assistant", "content": example["output_json"]},
    ]
    return {"text": tokenizer.apply_chat_template(messages, tokenize=False)}

train_ds = dataset["train"].map(format_chat)
val_ds = dataset["val"].map(format_chat)

# Run Trainer
trainer = SFTTrainer(
    model=model,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    peft_config=peft_config,
    dataset_text_field="text",
    max_seq_length=sft_cfg["training"]["max_seq_length"],
    args=SFTConfig(
        output_dir=sft_cfg["training"]["output_dir"],
        num_train_epochs=sft_cfg["training"]["epochs"],
        per_device_train_batch_size=sft_cfg["training"]["batch_size"],
        gradient_accumulation_steps=sft_cfg["training"]["grad_accum"],
        learning_rate=float(sft_cfg["training"]["learning_rate"]),
        logging_steps=10,
        eval_strategy="steps",
        eval_steps=50,
        save_strategy="epoch",
        fp16=True,
        report_to="none",
    ),
)

print("Starting SFT fine-tuning...")
trainer.train()

print(f"Saving SFT adapter...")
trainer.save_model(sft_cfg["training"]["output_dir"])
print("SFT Complete!")

## Stage 3: DPO Alignment (Preference Tuning)
We reload the base model in 4-bit, overlay our trained SFT adapter (as trainable), and apply DPO preferences.

In [ ]:
import torch
import yaml
from pathlib import Path
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel
from trl import DPOTrainer, DPOConfig
from src.config import load_config

cfg = load_config()
with open("configs/dpo.yaml", "r", encoding="utf-8") as f:
    dpo_cfg = yaml.safe_load(f)

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

print("Loading tokenizer and 4-bit base model...")
tokenizer = AutoTokenizer.from_pretrained(cfg.base_model)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

base_model = AutoModelForCausalLM.from_pretrained(
    cfg.base_model,
    quantization_config=bnb_config,
    device_map="auto",
)

# Load SFT adapter as active/trainable weights
sft_adapter_path = "artifacts/adapter_sft"
print(f"Loading SFT adapter from: {sft_adapter_path}")
model = PeftModel.from_pretrained(
    base_model,
    sft_adapter_path,
    is_trainable=True,
)

# Load DPO dataset
dpo_dataset_path = Path(cfg.data["dpo_out_dir"]) / "dpo_train.jsonl"
dataset = load_dataset("json", data_files={"train": str(dpo_dataset_path)})["train"]

# Format inputs with chat template
def format_dpo(example):
    messages = [
        {"role": "system", "content": cfg.system_prompt},
        {"role": "user", "content": example["prompt"]},
    ]
    prompt_text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    return {
        "prompt": prompt_text,
        "chosen": example["chosen"] + tokenizer.eos_token,
        "rejected": example["rejected"] + tokenizer.eos_token,
    }

dpo_dataset = dataset.map(format_dpo)

# Setup DPOTrainer
trainer = DPOTrainer(
    model=model,
    ref_model=None,
    train_dataset=dpo_dataset,
    tokenizer=tokenizer,
    args=DPOConfig(
        output_dir=dpo_cfg["training"]["output_dir"],
        beta=dpo_cfg["training"]["beta"],
        num_train_epochs=dpo_cfg["training"]["epochs"],
        per_device_train_batch_size=dpo_cfg["training"]["batch_size"],
        gradient_accumulation_steps=dpo_cfg["training"]["grad_accum"],
        learning_rate=float(dpo_cfg["training"]["learning_rate"]),
        max_length=dpo_cfg["training"]["max_seq_length"],
        max_prompt_length=256,
        logging_steps=10,
        save_strategy="epoch",
        fp16=True,
        report_to="none",
    ),
)

print("Starting DPO preference alignment...")
trainer.train()

print("Saving DPO adapter...")
trainer.save_model(dpo_cfg["training"]["output_dir"])
print("DPO alignment complete!")

## Stage 4: Merge LoRA Adapter
To create a standalone model (which llama.cpp needs to build a GGUF), we load the base model in unquantized FP16, apply our DPO adapter, and merge them.

In [ ]:
import torch
from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer
from src.config import load_config

cfg = load_config()
base_model_id = cfg.base_model
adapter_path = "artifacts/adapter_dpo"
merged_output_path = "artifacts/merged_model"

print(f"Loading base model in FP16 for merging: {base_model_id}")
tokenizer = AutoTokenizer.from_pretrained(base_model_id)
base_model = AutoModelForCausalLM.from_pretrained(
    base_model_id,
    torch_dtype=torch.float16,
    device_map="auto",
)

print(f"Loading active DPO adapter from: {adapter_path}")
model = PeftModel.from_pretrained(base_model, adapter_path)

print("Merging weights...")
merged_model = model.merge_and_unload()

print(f"Saving merged model to: {merged_output_path}")
merged_model.save_pretrained(merged_output_path)
tokenizer.save_pretrained(merged_output_path)
print("Weight merging complete!")

## Stage 5: GGUF Conversion & Quantization
Finally, compile llama.cpp, convert the merged model into GGUF format, and quantize it to 4-bit (`Q4_K_M`).

In [ ]:
# Clone llama.cpp and compile quantize tool
!git clone https://github.com/ggerganov/llama.cpp.git
!make -C llama.cpp
!pip install -r llama.cpp/requirements.txt

# Convert HF format to GGUF FP16
!python llama.cpp/convert_hf_to_gguf.py artifacts/merged_model --outfile artifacts/model-f16.gguf

# Quantize to Q4_K_M (approximately 400MB)
!./llama.cpp/llama-quantize artifacts/model-f16.gguf artifacts/model-Q4_K_M.gguf Q4_K_M